# Corpus Filtering, Sampling, and Gold Set Extraction

**Purpose:** Executes a memory-efficient streaming pipeline to filter parsed Stanza shards based on structural quality and linguistic complexity. Stratifies the filtered data to assemble the final 120K training corpus (50K `-լով`, 50K `-ած`, 20K both) and extracts a representative 2,000-sentence Gold benchmark set for human annotation.

**Inputs:**
- Unfiltered Stanza JSONL shards (`stanza_output/`).

**Outputs:**
- Quality-filtered intermediate JSONL records (`filtered_corpus_v2/kept_*.jsonl`).
- The final balanced 120K training corpus (`filtered_120k_combined.json`).
- The 2K Gold benchmark evaluation set in JSON, TXT, and XLSX formats (`gold_2k/`).

**Hardware Requirements:** CPU with standard RAM (streaming pipeline avoids memory overflow).

**Estimated Runtime:** ~5-10 minutes.

## Setup and Configuration

In [ ]:
import os
import json
import random
from collections import defaultdict, Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional

from tqdm.auto import tqdm

DATA_DIR = Path("./data")
STANZA_DIR = DATA_DIR / "stanza_output"
FILTERED_DIR = DATA_DIR / "filtered_corpus_v2"
GOLD_DIR = DATA_DIR / "gold_2k"

FILTERED_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_LOV = 50_000
TARGET_AC = 50_000
TARGET_BOTH = 20_000
TOTAL_CORPUS = TARGET_LOV + TARGET_AC + TARGET_BOTH
GOLD_SIZE = 2000

LOV_SUFFIX = "լով"
AC_SUFFIX = "ած"
ADDITIVE_DEPRELS = {"obj", "iobj", "obl", "advmod", "acl", "ccomp", "nsubj:pass"}
COORD_DEPRELS = {"cc", "conj"}

SEED = 42
random.seed(SEED)

## Phase 1: Structural Quality Filtering

In [ ]:
def is_true_participle(token: Dict) -> bool:
    """Validates verb categorization and expected suffix matching."""
    if token["upos"] != "VERB":
        return False
    return token["text"].endswith(LOV_SUFFIX) or token["text"].endswith(AC_SUFFIX)

def get_participle_type(token: Dict) -> str:
    """Categorizes the participle token type based on suffix."""
    if token["text"].endswith(LOV_SUFFIX):
        return "lov"
    elif token["text"].endswith(AC_SUFFIX):
        return "ats"
    return "unknown"

def count_additives(participle_token: Dict, all_tokens: List[Dict]) -> Tuple[int, List[Dict]]:
    """Counts syntactic dependents (additives) modifying the participle."""
    p_idx = participle_token["token_index"]
    additives = [t for t in all_tokens if t["head_index"] == p_idx and t["token_index"] != p_idx and t["dep"] in ADDITIVE_DEPRELS | COORD_DEPRELS]
    return len(additives), additives

def find_main_verb(tokens: List[Dict]) -> Optional[Dict]:
    """Identifies the primary root verb of the sequence."""
    return next((t for t in tokens if t["dep"] == "root"), None)

def classify_position(participle_token: Dict, additives: List[Dict], main_verb: Optional[Dict], all_tokens: List[Dict]) -> str:
    """Determines the relative syntactic position (PRE/INTRA/POST) of the participle phrase."""
    gov_idx = participle_token["head_index"]
    gov_verb = next((all_tokens[gov_idx] for _ in [1] if 0 <= gov_idx < len(all_tokens) and all_tokens[gov_idx]["upos"] == "VERB"), main_verb)
    
    if not gov_verb:
        return "UNKNOWN"

    p_idx = participle_token["token_index"]
    mv_idx = gov_verb["token_index"]
    phrase_indices = [p_idx] + [a["token_index"] for a in additives]
    phrase_start, phrase_end = min(phrase_indices), max(phrase_indices)
    
    subject = next((t for t in all_tokens if t["head_index"] == mv_idx and t["dep"] in ("nsubj", "nsubj:pass")), None)
    
    if phrase_end < mv_idx:
        return "INTRA" if subject and phrase_start > subject["token_index"] else "PRE"
    elif phrase_start > mv_idx:
        return "POST"
    else:
        return "INTRA" if p_idx < mv_idx and subject and phrase_start > subject["token_index"] else ("PRE" if p_idx < mv_idx else "POST")

def score_sentence(record: Dict) -> Dict:
    """Evaluates sequence quality and identifies specific grammatical rules (R1-R6)."""
    tokens = record.get("tokens", [])
    if not tokens or "stanza_error" in record:
        return {**record, "decision": "REJECT_MALFORMED", "quality_score": 0}
    
    word_count = len([t for t in tokens if t["upos"] != "PUNCT"])
    if word_count > 50 or word_count < 3:
        return {**record, "decision": "REJECT_LENGTH", "quality_score": 0}
    
    participles = []
    for token in tokens:
        if is_true_participle(token):
            additive_count, additive_tokens = count_additives(token, tokens)
            main_verb = find_main_verb(tokens)
            pos = classify_position(token, additive_tokens, main_verb, tokens) if additive_count > 0 and main_verb else "UNKNOWN"
            participles.append({
                "text": token["text"],
                "type": get_participle_type(token),
                "token_index": token["token_index"],
                "additive_count": additive_count,
                "position": pos,
                "rule_hint": None
            })
    
    has_apa = False
    has_rel_pron = False
    has_rule6 = False
    
    for token in tokens:
        if token["text"] == " ապա" and token["upos"] == "ADV" and token["dep"] == "advmod":
            for p in participles:
                if p["token_index"] > token["token_index"] and p["additive_count"] > 0:
                    has_apa, p["position"] = True, "INTRA"
        if token["text"] in {"որ", "որը", "որոնք"} and token["dep"] == "nsubj":
            for p in participles:
                if p["token_index"] > token["token_index"] and p["additive_count"] > 0:
                    has_rel_pron, p["position"] = True, "INTRA"
                    
    for p in [x for x in participles if x["additive_count"] > 0]:
        p_token = tokens[p["token_index"]]
        if p["type"] == "ats" and p_token["dep"] == "xcomp":
            has_rule6, p["rule_hint"] = True, "R6"
        elif p["type"] == "ats" and p_token["dep"] == "advcl":
            p["rule_hint"] = "R3"
            
    phrases = [p for p in participles if p["additive_count"] > 0]
    if not phrases:
        return {**record, "decision": "REJECT_LONE_PARTICIPLE", "quality_score": 0}
        
    score = 3
    if all(p["position"] != "UNKNOWN" for p in phrases): score += 2
    if len(phrases) >= 2: score += 2
    types = {p["type"] for p in participles}
    if "lov" in types and "ats" in types: score += 1
    if "stanza_error" not in record: score += 1
    if 10 <= word_count <= 30: score += 1
    if has_apa or has_rel_pron or has_rule6: score += 1
    if any(p["position"] == "UNKNOWN" for p in phrases): score -= 2
    if word_count > 40: score -= 2
    
    applicable_rules = set()
    for p in phrases:
        if has_apa and p["position"] == "INTRA": applicable_rules.add("R4")
        elif has_rel_pron and p["position"] == "INTRA": applicable_rules.add("R5")
        elif p.get("rule_hint") == "R6": applicable_rules.add("R6")
        elif p.get("rule_hint") == "R3": applicable_rules.add("R3")
        elif p["position"] == "PRE": applicable_rules.add("R2")
        elif p["position"] == "INTRA": applicable_rules.add("R1")
        elif p["position"] == "POST": applicable_rules.add("R3")
        
    pos_counts = Counter(p["position"] for p in phrases)
    
    return {
        **record,
        "decision": "KEEP" if max(0, min(10, score)) >= 5 else "REJECT_LOW_QUALITY",
        "quality_score": max(0, min(10, score)),
        "participles": participles,
        "sampling_tags": {
            "primary_position": max(pos_counts, key=pos_counts.get) if pos_counts else "UNKNOWN",
            "participle_type": "both" if ("lov" in types and "ats" in types) else ("lov_only" if "lov" in types else "ats_only"),
            "complexity": "compound" if sum(1 for t in tokens if t["dep"] in ("root", "conj") and t["upos"] == "VERB") > 1 else ("multi_phrase" if len(phrases) >= 2 else "single_phrase"),
            "applicable_rules": list(applicable_rules),
            "word_count": word_count
        }
    }

def filter_and_save_streaming(corpus_name: str):
    """Streams raw Stanza JSONL shards, applies scoring, and writes to filtered outputs."""
    shard_files = sorted(STANZA_DIR.glob(f"{corpus_name}_shard_*.jsonl"))
    outpath = FILTERED_DIR / f"kept_{corpus_name}.jsonl"
    
    with open(outpath, "w", encoding="utf-8") as out:
        for sf in tqdm(shard_files, desc=f"Filtering {corpus_name}"):
            with open(sf, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    record = json.loads(line.strip())
                    
                    if record.get("tokens"):
                        first_word = record.get("text", "").split()[0].strip("«»,՝:()—-")
                        first_token = record["tokens"][0]["text"].strip("«»,՝:()—-")
                        if first_word != first_token: continue
                        
                    res = score_sentence(record)
                    if res.get("decision") == "KEEP":
                        out.write(json.dumps(res, ensure_ascii=False) + "\n")

for corpus in ["lov", "ac", "both"]:
    if list(STANZA_DIR.glob(f"{corpus}_shard_*.jsonl")):
        filter_and_save_streaming(corpus)

## Phase 2: 120K Stratified Sampling

In [ ]:
def stream_and_sample(corpus_name: str, target_size: int) -> List[Dict]:
    """Samples the filtered corpus ensuring representation of advanced rules and positions."""
    filepath = FILTERED_DIR / f"kept_{corpus_name}.jsonl"
    advanced = []
    by_position = defaultdict(list)
    
    if not filepath.exists():
        return []
        
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            rules = record.get("sampling_tags", {}).get("applicable_rules", [])
            pos = record.get("sampling_tags", {}).get("primary_position", "UNKNOWN")
            
            if any(r in rules for r in ["R4", "R5", "R6"]):
                advanced.append(record)
            elif len(by_position[pos]) < target_size:
                by_position[pos].append(record)
                
    selected = list(advanced)
    remaining_quota = target_size - len(selected)
    
    if remaining_quota > 0:
        pos_quota = {
            "INTRA": int(remaining_quota * 0.40),
            "PRE":   int(remaining_quota * 0.30),
            "POST":  int(remaining_quota * 0.30),
        }
        pos_quota["INTRA"] += (remaining_quota - sum(pos_quota.values()))
        
        for pos, quota in pos_quota.items():
            selected.extend(by_position.get(pos, [])[:quota])
            
        if len(selected) < target_size:
            selected_ids = {id(s) for s in selected}
            all_remaining = [item for pool in by_position.values() for item in pool if id(item) not in selected_ids]
            selected.extend(all_remaining[:target_size - len(selected)])
            
    return selected[:target_size]

combined_corpus = []
for name, target in [("lov", TARGET_LOV), ("ac", TARGET_AC), ("both", TARGET_BOTH)]:
    sampled = stream_and_sample(name, target)
    combined_corpus.extend(sampled)

if len(combined_corpus) < TOTAL_CORPUS and list(FILTERED_DIR.glob("kept_ac.jsonl")):
    deficit = TOTAL_CORPUS - len(combined_corpus)
    existing_texts = {r["text"] for r in combined_corpus}
    extras = []
    with open(FILTERED_DIR / "kept_ac.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line.strip())
            if rec["text"] not in existing_texts:
                extras.append(rec)
                if len(extras) >= deficit: break
    combined_corpus.extend(extras)

random.shuffle(combined_corpus)
if combined_corpus:
    with open(FILTERED_DIR / "filtered_120k_combined.json", "w", encoding="utf-8") as f:
        json.dump(combined_corpus, f, ensure_ascii=False)
    print(f"Successfully saved filtered_120k_combined.json ({len(combined_corpus)} records).")

## Phase 3: 2K Gold Benchmark Extraction & Formatting

In [ ]:
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    OPENPYXL_AVAILABLE = True
except ImportError:
    OPENPYXL_AVAILABLE = False
    print("openpyxl not installed. XLSX generation will be skipped.")

if combined_corpus:
    for rec in combined_corpus:
        c = rec.get("corpus", "")
        rec["_source"] = "lov" if "lov" in c and "ac" not in c else ("ac" if "ac" in c and "lov" not in c else "both")
        
    quotas = {"lov": 833, "ac": 833, "both": 334}
    gold_2k = []

    for source, quota in quotas.items():
        pool = [r for r in combined_corpus if r.get("_source") == source]
        advanced = [r for r in pool if any(rule in r.get("sampling_tags", {}).get("applicable_rules", []) for rule in ["R4", "R5", "R6"])]
        core = [r for r in pool if r not in advanced]
        
        adv_quota = min(int(quota * 0.20), len(advanced))
        core_quota = quota - adv_quota
        
        random.shuffle(advanced)
        random.shuffle(core)
        gold_2k.extend(advanced[:adv_quota])
        gold_2k.extend(core[:core_quota])

    while len(gold_2k) < GOLD_SIZE:
        gold_2k.append(gold_2k[-1])
    gold_2k = gold_2k[:GOLD_SIZE]
    random.shuffle(gold_2k)

    with open(GOLD_DIR / "gold_2k_metadata.json", "w", encoding="utf-8") as f:
        json.dump(gold_2k, f, ensure_ascii=False, indent=2)
        
    with open(GOLD_DIR / "gold_2k_sentences.txt", "w", encoding="utf-8") as f:
        for i, rec in enumerate(gold_2k, 1):
            f.write(f"{i}. {rec['text']}\n")

    if OPENPYXL_AVAILABLE:
        wb = Workbook()
        ws = wb.active
        ws.title = "Gold 2K Annotation"
        
        headers = ["#", "Original Sentence", "Your Correction", "Notes", "Rule Hint", "Position", "Participle(s)", "Score", "Complexity"]
        header_fill = PatternFill("solid", fgColor="4472C4")
        header_font = Font(bold=True, color="FFFFFF", size=11)
        thin_border = Border(left=Side(style="thin"), right=Side(style="thin"), top=Side(style="thin"), bottom=Side(style="thin"))
        
        for col, header in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col, value=header)
            cell.fill, cell.font, cell.border = header_fill, header_font, thin_border
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
            
        for i, rec in enumerate(gold_2k, 1):
            row = i + 1
            tags = rec.get("sampling_tags", {})
            parts = rec.get("participles", [])
            
            ws.cell(row=row, column=1, value=i).border = thin_border
            ws.cell(row=row, column=2, value=rec["text"]).border = thin_border
            
            correction_cell = ws.cell(row=row, column=3, value="")
            correction_cell.fill = PatternFill("solid", fgColor="FFFFCC")
            correction_cell.border = thin_border
            
            ws.cell(row=row, column=4, value="").border = thin_border
            ws.cell(row=row, column=5, value=", ".join(tags.get("applicable_rules", []))).border = thin_border
            ws.cell(row=row, column=6, value=tags.get("primary_position", "?")).border = thin_border
            ws.cell(row=row, column=7, value=", ".join(f"{p['text']} ({p['type']})" for p in parts)).border = thin_border
            ws.cell(row=row, column=8, value=rec.get("quality_score", "?")).border = thin_border
            ws.cell(row=row, column=9, value=tags.get("complexity", "?")).border = thin_border
            
        ws.column_dimensions["B"].width = 80
        ws.column_dimensions["C"].width = 80
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:I{len(gold_2k) + 1}"
        
        wb.save(GOLD_DIR / "gold_2k_annotation.xlsx")